<a href="https://colab.research.google.com/github/niharikakt024/AI-Agent-for-Data-Cleaning/blob/main/DATA_CLEANING_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq pandas -q

In [ ]:
!pip install thefuzz python-Levenshtein -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in 5 words"}]
)
print(response.choices[0].message.content)

Hello, how are you today?


In [35]:
from google.colab import files
uploaded = files.upload()

Saving messy_employee_data.csv to messy_employee_data (1).csv


In [47]:
import pandas as pd

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"Shape: {df.shape}")
df.head()

Shape: (520, 7)


,EmployeeID,Name,Department,Gender,JoinDate,Age,Salary
0,1,Employee_1,sales,Male,20-Jul-2022,29.4,1883539.78
1,2,Employee_2,Sales,F,2023-10-28,35.2,98001.62
2,3,Employee_3,Sales,Male,08-22-2024,NaN,"$56,436.80"
3,4,Employee_4,sales,Female,07-Oct-2018,37.5,"$55,021.49"
4,5,Employee_5,Sales,F,11-30-2022,34.1,51123.83


In [48]:
def profile_dataframe(df):
    profile = {}
    for col in df.columns:
        profile[col] = {
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_pct": round(df[col].isna().mean() * 100, 2),
            "unique_count": int(df[col].nunique()),
            "sample_values": df[col].dropna().astype(str).unique()[:8].tolist()
        }
    return profile

profile = profile_dataframe(df)
for col, info in profile.items():
    print(col, "->", info)
    print()

EmployeeID -> {'dtype': 'int64', 'missing_count': 0, 'missing_pct': np.float64(0.0), 'unique_count': 500, 'sample_values': ['1', '2', '3', '4', '5', '6', '7', '8']}

Name -> {'dtype': 'object', 'missing_count': 0, 'missing_pct': np.float64(0.0), 'unique_count': 500, 'sample_values': ['Employee_1', 'Employee_2', 'Employee_3', 'Employee_4', 'Employee_5', 'Employee_6', 'Employee_7', 'Employee_8']}

Department -> {'dtype': 'object', 'missing_count': 26, 'missing_pct': np.float64(5.0), 'unique_count': 8, 'sample_values': ['sales', 'Sales', 'SALES', 'Marketing', 'engineering', 'Hr', 'Engineering', 'HR']}

Gender -> {'dtype': 'object', 'missing_count': 28, 'missing_pct': np.float64(5.38), 'unique_count': 6, 'sample_values': ['Male', 'F', 'Female', 'female', 'male', 'M']}

JoinDate -> {'dtype': 'object', 'missing_count': 0, 'missing_pct': np.float64(0.0), 'unique_count': 489, 'sample_values': ['20-Jul-2022', '2023-10-28', '08-22-2024', '07-Oct-2018', '11-30-2022', '22/09/2020', '2024-10-06', '

In [49]:
import json

def get_cleaning_plan(profile):
    prompt = f"""You are a data cleaning agent. Given this column profile from a pandas DataFrame,
decide the best cleaning action for each column.

PROFILE:
{json.dumps(profile, indent=2, default=str)}

For each column, return a JSON object with this exact structure (no markdown, no explanation outside the JSON):

{{
  "column_name": {{
    "issue": "short description of the problem",
    "action": "one of: standardize_case | parse_dates | clean_currency | fill_missing_median | fill_missing_mode | drop_duplicates | none",
    "reasoning": "one sentence why"
  }}
}}

Return ONLY valid JSON, nothing else."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    text = response.choices[0].message.content
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

plan = get_cleaning_plan(profile)
print(json.dumps(plan, indent=2))

{
  "EmployeeID": {
    "issue": "no issues found",
    "action": "none",
    "reasoning": "The column has no missing values and appears to be a unique identifier."
  },
  "Name": {
    "issue": "no issues found",
    "action": "none",
    "reasoning": "The column has no missing values and appears to be a unique identifier."
  },
  "Department": {
    "issue": "inconsistent casing and missing values",
    "action": "standardize_case",
    "reasoning": "Standardizing the case will help to reduce inconsistencies and prepare the data for further analysis or missing value imputation."
  },
  "Gender": {
    "issue": "inconsistent casing and missing values",
    "action": "standardize_case",
    "reasoning": "Standardizing the case will help to reduce inconsistencies and prepare the data for further analysis or missing value imputation."
  },
  "JoinDate": {
    "issue": "inconsistent date formats",
    "action": "parse_dates",
    "reasoning": "Parsing the dates will help to convert them i

In [50]:
import pandas as pd
import numpy as np

def apply_cleaning_plan(df, plan):
    df_clean = df.copy()
    log = []

    for col, spec in plan.items():
        action = spec.get("action", "").lower()
        if col not in df_clean.columns:
            continue

        if "standardize_case" in action:
            df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
            df_clean[col] = df_clean[col].replace("Nan", np.nan)
            log.append(col + ": standardized text casing")

        if "parse_dates" in action:
            df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce", format="mixed")
            log.append(col + ": parsed into consistent datetime format")

        if "clean_currency" in action:
            cleaned = df_clean[col].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
            df_clean[col] = pd.to_numeric(cleaned, errors="coerce")
            log.append(col + ": cleaned currency symbols, converted to numeric")

        if "fill_missing_median" in action:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
            log.append(col + ": filled missing values with median (" + str(round(median_val, 2)) + ")")

        if "fill_missing_mode" in action:
            mode_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(mode_val)
            log.append(col + ": filled missing values with mode (" + str(mode_val) + ")")

    return df_clean, log

df_clean, log = apply_cleaning_plan(df, plan)

print("=== CLEANING LOG ===")
for entry in log:
    print("-", entry)

print("\n=== BEFORE (missing values) ===")
print(df.isna().sum())

print("\n=== AFTER (missing values) ===")
print(df_clean.isna().sum())

df_clean.head(10)

=== CLEANING LOG ===
- Department: standardized text casing
- Gender: standardized text casing
- JoinDate: parsed into consistent datetime format
- Age: filled missing values with median (33.6)
- Salary: cleaned currency symbols, converted to numeric

=== BEFORE (missing values) ===
EmployeeID     0
Name           0
Department    26
Gender        28
JoinDate       0
Age           27
Salary        25
dtype: int64

=== AFTER (missing values) ===
EmployeeID     0
Name           0
Department    26
Gender        28
JoinDate       0
Age            0
Salary        25
dtype: int64


,EmployeeID,Name,Department,Gender,JoinDate,Age,Salary
0,1,Employee_1,Sales,Male,2022-07-20,29.4,1883539.78
1,2,Employee_2,Sales,F,2023-10-28,35.2,98001.62
2,3,Employee_3,Sales,Male,2024-08-22,33.6,56436.80
3,4,Employee_4,Sales,Female,2018-10-07,37.5,55021.49
4,5,Employee_5,Sales,F,2022-11-30,34.1,51123.83
5,6,Employee_6,Sales,Male,2020-09-22,29.1,61519.80
6,7,Employee_7,Marketing,Male,2024-10-06,50.3,84566.59
7,8,Employee_8,Sales,Female,2018-06-07,42.4,53205.36
8,9,Employee_9,Engineering,Female,2022-07-12,38.8,77520.21
9,10,Employee_10,Hr,Female,2020-04-14,26.5,82075.78


In [51]:
gender_map = {
    "M": "Male", "Male": "Male", "male": "Male",
    "F": "Female", "Female": "Female", "female": "Female"
}

df_clean["Gender"] = df["Gender"].astype(str).str.strip().map(gender_map)
df_clean["Gender"] = df_clean["Gender"].fillna(df_clean["Gender"].mode()[0])

dept_map = {
    "sales": "Sales", "Sales": "Sales", "SALES": "Sales",
    "marketing": "Marketing", "Marketing": "Marketing",
    "engineering": "Engineering", "Engineering": "Engineering",
    "hr": "HR", "Hr": "HR", "HR": "HR"
}
df_clean["Department"] = df["Department"].astype(str).str.strip().map(dept_map)
df_clean["Department"] = df_clean["Department"].fillna(df_clean["Department"].mode()[0])

print(df_clean["Gender"].value_counts())
print(df_clean["Department"].value_counts())

Gender
Female    283
Male      237
Name: count, dtype: int64
Department
Sales          208
HR             127
Engineering    123
Marketing       62
Name: count, dtype: int64


In [52]:
dupes = df_clean[df_clean.duplicated(subset=["EmployeeID"], keep=False)]
print(f"Duplicate rows found: {len(dupes)}")
dupes.sort_values("EmployeeID").head(10)

Duplicate rows found: 40


,EmployeeID,Name,Department,Gender,JoinDate,Age,Salary
27,28,Employee_28,HR,Female,2021-08-22,32.9,68499.73
507,28,Employee_28,HR,Female,2021-08-22,32.9,68499.73
502,38,Employee_38,HR,Female,2023-01-15,42.7,76322.55
37,38,Employee_38,HR,Female,2023-01-15,42.7,76322.55
87,88,Employee_88,Sales,Male,2018-05-16,44.1,92427.08
517,88,Employee_88,Sales,Male,2018-05-16,44.1,92427.08
518,113,Employee_113,Engineering,Male,2023-03-18,28.3,43293.99
112,113,Employee_113,Engineering,Male,2023-03-18,28.3,43293.99
130,131,Employee_131,HR,Female,2024-10-13,23.1,83353.07
500,131,Employee_131,HR,Female,2024-10-13,23.1,83353.07


In [53]:
df_clean = df.copy()

In [55]:
before_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["EmployeeID"], keep="first")
after_count = len(df_clean)

print(f"Removed {before_count - after_count} duplicate rows")
print(f"Final shape: {df_clean.shape}")

Removed 20 duplicate rows
Final shape: (500, 7)


In [56]:
def generate_report(df_before, df_after, log, dupes_removed):
    report_lines = []
    report_lines.append("=" * 50)
    report_lines.append("DATA CLEANING AGENT — SUMMARY REPORT")
    report_lines.append("=" * 50)
    report_lines.append(f"\nOriginal rows: {len(df_before)}")
    report_lines.append(f"Final rows: {len(df_after)}")
    report_lines.append(f"Duplicate rows removed: {dupes_removed}")
    report_lines.append(f"\nTotal missing values before: {df_before.isna().sum().sum()}")
    report_lines.append(f"Total missing values after: {df_after.isna().sum().sum()}")
    report_lines.append("\nActions taken:")
    for entry in log:
        report_lines.append(f"  - {entry}")
    report_lines.append(f"  - Gender standardized to Male/Female")
    report_lines.append(f"  - Department standardized to Sales/Marketing/Engineering/HR")
    report_lines.append(f"  - {dupes_removed} exact duplicate rows removed")
    report_lines.append("\n" + "=" * 50)
    return "\n".join(report_lines)

report = generate_report(df, df_clean, log, before_count - after_count)
print(report)

# Save the cleaned data + report
df_clean.to_csv("cleaned_data.csv", index=False)
with open("cleaning_report.txt", "w") as f:
    f.write(report)

from google.colab import files
files.download("cleaned_data.csv")
files.download("cleaning_report.txt")

DATA CLEANING AGENT — SUMMARY REPORT

Original rows: 520
Final rows: 500
Duplicate rows removed: 20

Total missing values before: 106
Total missing values after: 100

Actions taken:
  - Department: standardized text casing
  - Gender: standardized text casing
  - JoinDate: parsed into consistent datetime format
  - Age: filled missing values with median (33.6)
  - Salary: cleaned currency symbols, converted to numeric
  - Gender standardized to Male/Female
  - Department standardized to Sales/Marketing/Engineering/HR
  - 20 exact duplicate rows removed



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>